[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/100.%20The%20Closed-Loop%20%28Reverse%20Logistics%29%20Network%20Design%20Problem/P100-Tier-11_executed.ipynb)

---


# 100. The Closed-Loop Network Design Problem
## Tier 11: Physical-Cyber Synthesis in a Warehouse

### Purpose and scope

This notebook builds an illustrative simulation of a warehouse physical-cyber environment. It represents sensors, rule-based AI agents, state synchronization, performance metrics, and reporting in Python. The program is a systems simulation; it does **not** formulate or solve a closed-loop logistics network design optimization problem. It also does not connect to real warehouse equipment, IoT devices, autonomous services, or mixed-reality interfaces.

The simulation uses synthetic sensor readings and heuristic agent decisions to demonstrate how these components can be represented and inspected in code. The reported metrics are illustrative indicators defined by this model, not measured operational performance or validated predictions.

### Inputs and outputs

- **Inputs:** a fixed random seed, synthetic sensor and agent configurations, and a simulation horizon of three simulated minutes.
- **Process:** generate sensor values, calculate a synchronization indicator, provide a shared context to active agents, update performance metrics, and collect the results at each time step.
- **Outputs:** time-series data, a summary report, plots, and descriptive statistics.

### Model assumptions

- The warehouse is represented by a 100-by-100-by-10 coordinate region. Sensor and agent locations are stored but do not affect decisions or metrics.
- Sensor readings are synthetic. The model adds random variation and simple noise; it does not model sensor calibration, communication delays, failures, or spatial coverage.
- Agent actions are selected by fixed threshold rules. Agents do not communicate with each other, execute actions, learn from outcomes, or optimize a network.
- Synchronization status is a constructed indicator based on the mean accuracy of active sensors. It is not measured latency or a comparison between independently observed physical and digital states.
- The simulation advances in one-second model-time steps and runs faster than real time. The short sleep is only a pacing delay.
- Security, costs, reverse-logistics flows, facility location, capacity, and environmental impacts are outside the scope of this example.

### Approach

1. Define sensor, agent, and environment data structures.
2. Create a synthetic warehouse environment with 1,000 sensors and a set of agents.
3. Advance the environment through sensor updates, synchronization calculations, heuristic decisions, and metric collection.
4. Visualize the collected time series and summarize the model-defined indicators.

### How to interpret the results

Use the outputs to understand the behavior of this particular simulation and how its code components interact. A high synchronization indicator means that the model's active sensors have high assigned accuracy; it does not establish that a real digital twin is synchronized. Likewise, the system-health score combines three model metrics for illustration and is not an industry standard. Conclusions about real operations require operational data, validated definitions, and an appropriate network-design model.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from enum import Enum
import random
import time
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# A fixed seed makes the synthetic random draws reproducible in a fresh run.
np.random.seed(42)
random.seed(42)

In [ ]:
class AgentType(Enum):
    """Roles used by the heuristic agents in the simulation."""

    COORDINATOR = "coordinator"
    OPERATOR = "operator"
    OPTIMIZER = "optimizer"
    ANALYST = "analyst"
    INTERFACE = "interface"


@dataclass
class MetaverseSensor:
    """Synthetic sensor with a location, type, reading, and assigned accuracy."""

    sensor_id: str
    location: Tuple[float, float, float]
    sensor_type: str
    value: float = 0.0
    last_update: float = 0.0
    accuracy: float = 0.95
    status: str = "active"

    def update_value(self, new_value: float, timestamp: float) -> None:
        """Store a new synthetic reading and its model-time timestamp."""
        self.value = new_value + np.random.normal(0, 0.05)
        self.last_update = timestamp


@dataclass
class AIAgent:
    """Rule-based agent that records decisions from a shared context."""

    agent_id: str
    agent_type: AgentType
    capabilities: List[str] = field(default_factory=list)
    location: Tuple[float, float, float] = (0.0, 0.0, 0.0)
    status: str = "active"
    performance_score: float = 0.8
    decision_history: List[Dict[str, Any]] = field(default_factory=list)
    learning_rate: float = 0.1

    def make_decision(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """Apply the rule associated with this agent's role and record it."""
        decision_functions = {
            AgentType.COORDINATOR: self._coordinate_decision,
            AgentType.OPERATOR: self._operate_decision,
            AgentType.OPTIMIZER: self._optimize_decision,
            AgentType.ANALYST: self._analyze_decision,
            AgentType.INTERFACE: self._interface_decision,
        }
        decision = decision_functions[self.agent_type](context)
        self.decision_history.append(decision)
        return decision

    def _make_decision_record(
        self, action: str, confidence: float, reasoning: str
    ) -> Dict[str, Any]:
        return {
            "agent_id": self.agent_id,
            "timestamp": time.time(),
            "decision_type": self.agent_type.value,
            "action": action,
            "confidence": confidence,
            "reasoning": reasoning,
        }

    def _coordinate_decision(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """Choose a coordination action from the system-status label."""
        system_status = context.get("system_status", "normal")
        if system_status == "overloaded":
            action, confidence = "rebalance_load", 0.9
            reasoning = "System overloaded; redistribute tasks among agents."
        elif system_status == "underutilized":
            action, confidence = "consolidate_resources", 0.8
            reasoning = "System underutilized; consolidate resources for efficiency."
        else:
            action, confidence = "maintain_balance", 0.7
            reasoning = "System status does not trigger a special coordination rule."
        return self._make_decision_record(action, confidence, reasoning)

    def _operate_decision(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """Choose an operational action from the task-queue length."""
        task_queue = context.get("task_queue", [])
        if len(task_queue) > 10:
            action, confidence = "increase_capacity", 0.8
            reasoning = "The task queue is long; increase operational capacity."
        elif len(task_queue) < 3:
            action, confidence = "optimize_efficiency", 0.7
            reasoning = "The task queue is short; optimize operational efficiency."
        else:
            action, confidence = "maintain_current", 0.6
            reasoning = "The task queue is moderate; maintain current operations."
        return self._make_decision_record(action, confidence, reasoning)

    def _optimize_decision(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """Choose a monitoring or tuning action from reported efficiency."""
        metrics = context.get("performance_metrics", {})
        efficiency = metrics.get("efficiency", 0.8)
        if efficiency < 0.7:
            action, confidence = "reconfigure_network", 0.9
            reasoning = "Efficiency is below the rule threshold; reconfigure the network."
        elif efficiency > 0.9:
            action, confidence = "fine_tune_parameters", 0.7
            reasoning = "Efficiency is high; fine-tune parameters."
        else:
            action, confidence = "monitor_performance", 0.6
            reasoning = "Efficiency is within the monitoring range."
        return self._make_decision_record(action, confidence, reasoning)

    def _analyze_decision(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """Choose an analysis action from data volume and anomaly count."""
        data_volume = context.get("data_volume", 1000)
        anomaly_count = context.get("anomaly_count", 0)
        if anomaly_count > 10:
            action, confidence = "investigate_anomalies", 0.9
            reasoning = "The anomaly count is high; investigate potential issues."
        elif data_volume > 5000:
            action, confidence = "analyze_patterns", 0.8
            reasoning = "The data volume is high; analyze patterns."
        else:
            action, confidence = "generate_report", 0.7
            reasoning = "Generate an analytical report under the current conditions."
        return self._make_decision_record(action, confidence, reasoning)

    def _interface_decision(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """Choose an interface action from activity and interface load."""
        user_activity = context.get("user_activity", "low")
        interface_load = context.get("interface_load", 0.5)
        if user_activity == "high":
            action, confidence = "enhance_interface", 0.8
            reasoning = "User activity is high; enhance interface responsiveness."
        elif interface_load > 0.8:
            action, confidence = "optimize_rendering", 0.7
            reasoning = "Interface load is high; optimize rendering performance."
        else:
            action, confidence = "maintain_interface", 0.6
            reasoning = "Maintain the interface under the current conditions."
        return self._make_decision_record(action, confidence, reasoning)

    def update_performance(self, outcome: float) -> None:
        """Update the score using an exponential moving-average rule."""
        self.performance_score = (
            (1 - self.learning_rate) * self.performance_score
            + self.learning_rate * outcome
        )


@dataclass
class MetaverseEnvironment:
    """Container for synthetic warehouse sensors, agents, and metrics."""

    environment_id: str
    sensors: List[MetaverseSensor] = field(default_factory=list)
    agents: List[AIAgent] = field(default_factory=list)
    physical_state: Dict[str, Any] = field(default_factory=dict)
    virtual_state: Dict[str, Any] = field(default_factory=dict)
    synchronization_status: float = 1.0
    performance_metrics: Dict[str, float] = field(default_factory=dict)

    def add_sensor(self, sensor: MetaverseSensor) -> None:
        self.sensors.append(sensor)

    def add_agent(self, agent: AIAgent) -> None:
        self.agents.append(agent)

    def update_sensors(self, timestamp: float) -> None:
        """Generate one synthetic reading for each active sensor."""
        for sensor in self.sensors:
            if sensor.status != "active":
                continue
            if sensor.sensor_type == "temperature":
                base_value = 20 + 10 * np.sin(timestamp / 3600 * 2 * np.pi)
                new_value = base_value + np.random.normal(0, 2)
            elif sensor.sensor_type == "humidity":
                base_value = 50 + 20 * np.sin(
                    timestamp / 3600 * 2 * np.pi + np.pi / 4
                )
                new_value = base_value + np.random.normal(0, 5)
            elif sensor.sensor_type == "motion":
                new_value = np.random.uniform(0, 100)
            elif sensor.sensor_type == "inventory":
                new_value = np.random.uniform(0, 1000)
            elif sensor.sensor_type == "traffic":
                new_value = np.random.uniform(0, 50)
            else:
                new_value = np.random.uniform(0, 100)
            sensor.update_value(new_value, timestamp)

    def synchronize_states(self) -> float:
        """Calculate the model's accuracy-based synchronization indicator."""
        active_sensors = [s for s in self.sensors if s.status == "active"]
        if not active_sensors:
            self.synchronization_status = 0.0
            return self.synchronization_status

        average_accuracy = np.mean([s.accuracy for s in active_sensors])
        average_state_difference = np.mean(
            [(1 - sensor.accuracy) * 0.1 for sensor in active_sensors]
        )
        self.synchronization_status = average_accuracy - average_state_difference
        return self.synchronization_status

    def run_agent_decisions(self) -> List[Dict[str, Any]]:
        """Build one shared context and collect decisions from active agents."""
        context = {
            "system_status": self._calculate_system_status(),
            "agent_status": {a.agent_id: a.status for a in self.agents},
            "task_queue": self._get_task_queue(),
            "resource_status": self._get_resource_status(),
            "performance_metrics": self.performance_metrics,
            "data_volume": len(self.sensors) * 100,
            "anomaly_count": self._count_anomalies(),
            "user_activity": self._get_user_activity(),
            "interface_load": self._get_interface_load(),
        }
        return [
            agent.make_decision(context)
            for agent in self.agents
            if agent.status == "active"
        ]

    def _calculate_system_status(self) -> str:
        """Classify system status from the share of active agents."""
        active_agents = sum(agent.status == "active" for agent in self.agents)
        total_agents = len(self.agents)
        if total_agents == 0:
            return "unknown"
        agent_ratio = active_agents / total_agents
        if agent_ratio > 0.8:
            return "normal"
        if agent_ratio > 0.5:
            return "degraded"
        return "critical"

    def _get_task_queue(self) -> List[str]:
        """Create a simplified task queue from active sensor count."""
        active_sensors = sum(sensor.status == "active" for sensor in self.sensors)
        return [f"task_{i}" for i in range(int(active_sensors * 0.5))]

    def _get_resource_status(self) -> Dict[str, float]:
        """Return synthetic resource-use indicators."""
        return {
            "cpu_usage": np.random.uniform(0.3, 0.8),
            "memory_usage": np.random.uniform(0.4, 0.7),
            "network_usage": np.random.uniform(0.2, 0.6),
            "storage_usage": np.random.uniform(0.5, 0.9),
        }

    def _count_anomalies(self) -> int:
        """Count active temperature and humidity readings outside fixed bounds."""
        count = 0
        for sensor in self.sensors:
            if sensor.status != "active":
                continue
            if sensor.sensor_type == "temperature" and not 10 <= sensor.value <= 35:
                count += 1
            elif sensor.sensor_type == "humidity" and not 20 <= sensor.value <= 80:
                count += 1
        return count

    def _get_user_activity(self) -> str:
        """Generate a synthetic user-activity category."""
        activity_level = np.random.random()
        if activity_level > 0.7:
            return "high"
        if activity_level > 0.3:
            return "medium"
        return "low"

    def _get_interface_load(self) -> float:
        return float(np.random.uniform(0.2, 0.9))

    def update_performance_metrics(self) -> None:
        """Recompute the model-defined metrics from current environment state."""
        active_agents = [a for a in self.agents if a.status == "active"]
        active_sensors = [s for s in self.sensors if s.status == "active"]
        average_performance = (
            np.mean([a.performance_score for a in active_agents])
            if active_agents else 0.0
        )
        self.performance_metrics = {
            "efficiency": float(average_performance),
            "synchronization": float(self.synchronization_status),
            "sensor_coverage": len(active_sensors) / len(self.sensors)
            if self.sensors else 0.0,
            "agent_utilization": len(active_agents) / len(self.agents)
            if self.agents else 0.0,
            "data_quality": float(np.mean([s.accuracy for s in active_sensors]))
            if active_sensors else 0.0,
            "response_time": float(np.random.uniform(0.1, 2.0)),
            "throughput": float(np.random.uniform(100, 1000)),
        }

    def generate_metaverse_report(self) -> Dict[str, Any]:
        """Summarize counts, metrics, and rule-based recommendations."""
        active_sensors = [s for s in self.sensors if s.status == "active"]
        active_agents = [a for a in self.agents if a.status == "active"]
        report = {
            "report_timestamp": time.time(),
            "environment_id": self.environment_id,
            "system_overview": {
                "total_sensors": len(self.sensors),
                "active_sensors": len(active_sensors),
                "total_agents": len(self.agents),
                "active_agents": len(active_agents),
                "synchronization_status": self.synchronization_status,
                "system_status": self._calculate_system_status(),
            },
            "sensor_analysis": {},
            "agent_analysis": {},
            "performance_metrics": self.performance_metrics.copy(),
            "recommendations": [],
        }

        sensor_types = defaultdict(int)
        for sensor in self.sensors:
            sensor_types[sensor.sensor_type] += 1
        report["sensor_analysis"] = {
            "sensor_types": dict(sensor_types),
            "average_accuracy": float(np.mean([s.accuracy for s in active_sensors]))
            if active_sensors else 0.0,
            "coverage_area": len(self.sensors) * 100,
        }

        agent_types = defaultdict(int)
        for agent in self.agents:
            agent_types[agent.agent_type.value] += 1
        report["agent_analysis"] = {
            "agent_types": dict(agent_types),
            "average_performance": float(
                np.mean([a.performance_score for a in self.agents])
            ) if self.agents else 0.0,
            "decision_frequency": sum(
                len(agent.decision_history) for agent in self.agents
            ) / len(self.agents) if self.agents else 0.0,
        }

        if self.synchronization_status < 0.8:
            report["recommendations"].append(
                "Review the model's physical-cyber synchronization indicator."
            )
        if len(active_sensors) < len(self.sensors) * 0.8:
            report["recommendations"].append(
                "Review sensor availability in the simulated environment."
            )
        if self.performance_metrics.get("efficiency", 0) < 0.7:
            report["recommendations"].append(
                "Review agent performance scores and coordination rules."
            )
        return report

In [ ]:
# Create the synthetic warehouse environment and its sensor population.
print("Initializing the synthetic warehouse environment...")
print("=" * 50)

metaverse = MetaverseEnvironment(environment_id="warehouse_metaverse_001")
sensor_types = ["temperature", "humidity", "motion", "inventory", "traffic"]
num_sensors_per_type = 200

print(f"Creating {len(sensor_types) * num_sensors_per_type} sensors...")
for sensor_type in sensor_types:
    for i in range(num_sensors_per_type):
        sensor = MetaverseSensor(
            sensor_id=f"{sensor_type}_{i}",
            location=(
                np.random.uniform(0, 100),
                np.random.uniform(0, 100),
                np.random.uniform(0, 10),
            ),
            sensor_type=sensor_type,
            accuracy=np.random.uniform(0.9, 0.99),
        )
        metaverse.add_sensor(sensor)

print(f"Sensors created: {len(metaverse.sensors)}")
for sensor_type in sensor_types:
    count = sum(sensor.sensor_type == sensor_type for sensor in metaverse.sensors)
    print(f"  {sensor_type.title()}: {count} sensors")

agent_configurations = [
    (AgentType.COORDINATOR, 5),
    (AgentType.OPERATOR, 10),
    (AgentType.OPTIMIZER, 8),
    (AgentType.ANALYST, 6),
    (AgentType.INTERFACE, 4),
]
print("\nCreating heuristic agents...")
for agent_type, count in agent_configurations:
    for i in range(count):
        agent = AIAgent(
            agent_id=f"{agent_type.value}_{i}",
            agent_type=agent_type,
            capabilities=[f"{agent_type.value}_capability_{j}" for j in range(3)],
            location=(
                np.random.uniform(0, 100),
                np.random.uniform(0, 100),
                np.random.uniform(0, 10),
            ),
            performance_score=np.random.uniform(0.7, 0.95),
        )
        metaverse.add_agent(agent)

print(f"Agents created: {len(metaverse.agents)}")
for agent_type, count in agent_configurations:
    print(f"  {agent_type.value.title()}: {count} agents")

metaverse.update_performance_metrics()
print("\nInitial model-defined performance metrics:")
for metric, value in metaverse.performance_metrics.items():
    print(f"  {metric.replace('_', ' ').title()}: {value:.3f}")

print("\nEnvironment ready:")
print(f"  Environment ID: {metaverse.environment_id}")
print(f"  Total sensors: {len(metaverse.sensors)}")
print(f"  Total agents: {len(metaverse.agents)}")
print(f"  Initial synchronization indicator: {metaverse.synchronization_status:.3f}")

In [ ]:
def run_metaverse_simulation(
    metaverse: MetaverseEnvironment, duration_minutes: int = 5
) -> Dict[str, List[Any]]:
    """Run one-second model-time steps and collect the resulting time series.

    The duration is simulated time. A short wall-clock sleep paces execution,
    so the simulation is intentionally accelerated rather than real-time.
    """
    if duration_minutes <= 0:
        raise ValueError("duration_minutes must be a positive integer.")

    print(f"Running the simulation for {duration_minutes} simulated minutes...")
    print("=" * 60)
    simulation_data = {
        "timestamps": [],
        "synchronization_status": [],
        "active_sensors": [],
        "active_agents": [],
        "performance_metrics": [],
        "agent_decisions": [],
        "anomaly_counts": [],
    }

    start_wall_time = time.time()
    model_start_time = start_wall_time
    for minute in range(duration_minutes):
        for second in range(60):
            # Model time advances by exactly one second per iteration, regardless
            # of how long the Python calculations take on the host machine.
            current_time = model_start_time + minute * 60 + second
            metaverse.update_sensors(current_time)
            sync_status = metaverse.synchronize_states()
            decisions = metaverse.run_agent_decisions()
            metaverse.update_performance_metrics()

            active_sensors = sum(
                sensor.status == "active" for sensor in metaverse.sensors
            )
            active_agents = sum(
                agent.status == "active" for agent in metaverse.agents
            )
            simulation_data["timestamps"].append(current_time)
            simulation_data["synchronization_status"].append(sync_status)
            simulation_data["active_sensors"].append(active_sensors)
            simulation_data["active_agents"].append(active_agents)
            simulation_data["performance_metrics"].append(
                metaverse.performance_metrics.copy()
            )
            simulation_data["agent_decisions"].append(decisions)
            simulation_data["anomaly_counts"].append(metaverse._count_anomalies())

            if second == 0:
                print(
                    f"Minute {minute + 1}: synchronization={sync_status:.3f}, "
                    f"active sensors={active_sensors}, active agents={active_agents}"
                )
                if decisions:
                    print(f"  Decisions collected: {len(decisions)}")
                    for decision in decisions[:3]:
                        print(
                            f"    {decision['agent_id']}: {decision['action']} "
                            f"(confidence: {decision['confidence']:.2f})"
                        )
            time.sleep(0.01)

    elapsed_wall_time = time.time() - start_wall_time
    print("\nSimulation summary:")
    print(f"Wall-clock runtime: {elapsed_wall_time:.2f} seconds")
    print(f"Model updates: {len(simulation_data['timestamps'])}")
    print(
        "Average synchronization indicator: "
        f"{np.mean(simulation_data['synchronization_status']):.3f}"
    )
    print(f"Average active sensors: {np.mean(simulation_data['active_sensors']):.1f}")
    print(f"Average active agents: {np.mean(simulation_data['active_agents']):.1f}")
    print(
        "Decision records: "
        f"{sum(len(items) for items in simulation_data['agent_decisions'])}"
    )
    print(f"Average anomaly count: {np.mean(simulation_data['anomaly_counts']):.1f}")
    return simulation_data


simulation_data = run_metaverse_simulation(metaverse, duration_minutes=3)
final_report = metaverse.generate_metaverse_report()

print("\nFinal model report:")
print("=" * 30)
overview = final_report["system_overview"]
print("System overview:")
print(f"  Total sensors: {overview['total_sensors']}")
print(
    f"  Active sensors: {overview['active_sensors']} "
    f"({overview['active_sensors'] / overview['total_sensors']:.1%})"
)
print(f"  Total agents: {overview['total_agents']}")
print(
    f"  Active agents: {overview['active_agents']} "
    f"({overview['active_agents'] / overview['total_agents']:.1%})"
)
print(f"  Synchronization indicator: {overview['synchronization_status']:.3f}")
print(f"  Agent-availability status: {overview['system_status']}")
print("\nRule-based recommendations:")
if final_report["recommendations"]:
    for index, recommendation in enumerate(final_report["recommendations"], 1):
        print(f"  {index}. {recommendation}")
else:
    print("  No recommendation thresholds were triggered.")

In [ ]:
def visualize_metaverse_performance(
    metaverse: MetaverseEnvironment,
    simulation_data: Dict[str, List[Any]],
) -> Dict[str, Any]:
    """Plot the collected indicators and print descriptive summaries."""
    timestamps = simulation_data["timestamps"]
    if not timestamps:
        raise ValueError("simulation_data must contain at least one time step.")

    time_minutes = [(timestamp - timestamps[0]) / 60 for timestamp in timestamps]
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    ax1.set_title("Physical-Cyber Synchronization Indicator", fontsize=14)
    ax1.plot(
        time_minutes,
        simulation_data["synchronization_status"],
        color="blue",
        linewidth=2,
        label="Model indicator",
    )
    ax1.axhline(
        y=0.8, color="red", linestyle="--", alpha=0.7,
        label="Illustrative reference: 0.8",
    )
    ax1.axhline(
        y=0.9, color="green", linestyle="--", alpha=0.7,
        label="Illustrative reference: 0.9",
    )
    ax1.set_xlabel("Simulated time (minutes)")
    ax1.set_ylabel("Synchronization indicator (unitless)")
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_title("Active Sensors and Agents", fontsize=14)
    sensor_line, = ax2.plot(
        time_minutes, simulation_data["active_sensors"],
        color="blue", label="Active sensors", linewidth=2,
    )
    ax2.set_xlabel("Simulated time (minutes)")
    ax2.set_ylabel("Active sensors (count)", color="blue")
    ax2.tick_params(axis="y", labelcolor="blue")
    ax2_twin = ax2.twinx()
    agent_line, = ax2_twin.plot(
        time_minutes, simulation_data["active_agents"],
        color="red", label="Active agents", linewidth=2,
    )
    ax2_twin.set_ylabel("Active agents (count)", color="red")
    ax2_twin.tick_params(axis="y", labelcolor="red")
    ax2.legend([sensor_line, agent_line], ["Active sensors", "Active agents"])
    ax2.grid(True, alpha=0.3)

    ax3.set_title("Model-Defined Performance Indicators", fontsize=14)
    efficiency = [m.get("efficiency", 0) for m in simulation_data["performance_metrics"]]
    data_quality = [m.get("data_quality", 0) for m in simulation_data["performance_metrics"]]
    ax3.plot(time_minutes, efficiency, color="green", label="Efficiency score")
    ax3.plot(time_minutes, data_quality, color="orange", label="Mean sensor accuracy")
    ax3.set_xlabel("Simulated time (minutes)")
    ax3.set_ylabel("Score (unitless)")
    ax3.set_ylim(0, 1)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ax4.set_title("Recorded Decisions by Agent Role", fontsize=14)
    decision_types = defaultdict(int)
    for decisions in simulation_data["agent_decisions"]:
        for decision in decisions:
            decision_types[decision["decision_type"]] += 1
    if decision_types:
        types = list(decision_types)
        counts = [decision_types[agent_type] for agent_type in types]
        bars = ax4.bar(types, counts, alpha=0.7, color="lightblue")
        ax4.set_xlabel("Agent role")
        ax4.set_ylabel("Decision records (count)")
        ax4.tick_params(axis="x", labelrotation=45)
        ax4.grid(True, axis="y", alpha=0.3)
        label_offset = max(counts) * 0.01
        for bar, count in zip(bars, counts):
            ax4.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + label_offset,
                str(count),
                ha="center",
                va="bottom",
                fontsize=9,
            )
    else:
        ax4.text(0.5, 0.5, "No decisions recorded", ha="center", va="center")
        ax4.set_axis_off()

    fig.tight_layout()
    plt.show()

    average_sync = float(np.mean(simulation_data["synchronization_status"]))
    average_efficiency = float(np.mean(efficiency))
    average_data_quality = float(np.mean(data_quality))
    system_health = (average_sync + average_efficiency + average_data_quality) / 3

    print("\nDescriptive performance summary:")
    print("=" * 35)
    print(f"Composite illustrative score: {system_health:.3f}")
    print(f"  Mean synchronization indicator: {average_sync:.3f}")
    print(f"  Mean efficiency score: {average_efficiency:.3f}")
    print(f"  Mean sensor-accuracy score: {average_data_quality:.3f}")

    agent_performance = {
        agent.agent_id: agent.performance_score for agent in metaverse.agents
    }
    if agent_performance:
        best_agent = max(agent_performance, key=agent_performance.get)
        worst_agent = min(agent_performance, key=agent_performance.get)
        print("\nAgent score summary:")
        print(f"  Highest score: {best_agent} ({agent_performance[best_agent]:.3f})")
        print(f"  Lowest score: {worst_agent} ({agent_performance[worst_agent]:.3f})")
        print(
            "  Mean score: "
            f"{np.mean(list(agent_performance.values())):.3f}"
        )

    anomaly_counts = simulation_data["anomaly_counts"]
    average_anomalies = float(np.mean(anomaly_counts))
    maximum_anomalies = max(anomaly_counts)
    sensor_count = len(metaverse.sensors)
    print("\nSynthetic anomaly summary:")
    print(f"  Mean flagged readings per time step: {average_anomalies:.1f}")
    print(f"  Maximum flagged readings in a time step: {maximum_anomalies}")
    print(
        "  Mean flagged-reading share of all sensors: "
        f"{average_anomalies / sensor_count * 100:.2f}%"
        if sensor_count else "  No sensors are configured."
    )

    total_decisions = sum(
        len(decisions) for decisions in simulation_data["agent_decisions"]
    )
    elapsed_simulated_minutes = (timestamps[-1] - timestamps[0]) / 60
    decisions_per_minute = (
        total_decisions / elapsed_simulated_minutes
        if elapsed_simulated_minutes > 0 else 0.0
    )
    agent_count = len(metaverse.agents)
    print("\nDecision-record summary:")
    print(f"  Total decision records: {total_decisions}")
    print(f"  Records per simulated minute: {decisions_per_minute:.1f}")
    print(
        "  Records per agent per simulated minute: "
        f"{decisions_per_minute / agent_count:.2f}"
        if agent_count else "  No agents are configured."
    )
    print("\nInterpretation:")
    if system_health > 0.85:
        print("  The composite score is high under the illustrative model thresholds.")
    elif system_health > 0.75:
        print("  The composite score is moderate-to-high under the illustrative thresholds.")
    elif system_health > 0.65:
        print("  The composite score is moderate under the illustrative thresholds.")
    else:
        print("  The composite score is low under the illustrative model thresholds.")
    print("  These labels describe this simulation only; they are not operational ratings.")

    return simulation_data


visualize_metaverse_performance(metaverse, simulation_data)

## Interpreting the model

The program combines several illustrative components: synthetic sensing, a shared context for rule-based agents, an accuracy-based synchronization indicator, and descriptive reporting. These components demonstrate a possible software representation of a physical-cyber system; they do not establish that the system is autonomous, self-organizing, or suitable for deployment.

The agent rules are deterministic given their context, but some context fields and sensor values are randomly generated. The fixed seed supports reproducibility when the notebook is run from a fresh kernel with the same code and software environment. The agents record proposed actions; the simulation does not execute those actions or update agent performance from outcomes. The `update_performance` method is defined for illustration but is not called by the simulation.

The plotted reference levels and composite score thresholds are descriptive choices within this example, not validated benchmarks. In particular, the composite score is the arithmetic mean of the mean synchronization indicator, efficiency score, and mean sensor-accuracy score. It should not be interpreted as a general measure of warehouse performance.

## Limitations and possible extensions

The model omits physical facility constraints, reverse-product flows, network design decisions, costs, capacities, service requirements, sensor communication delays, cybersecurity, and validation against operational data. Adding such features would require explicit data and carefully defined decision variables, constraints, and performance measures. This simulation alone does not solve those design questions.